# Privacy-Preserving Fraud Detection Using Machine Learning
### Academic Assessment: IA 2 — Implementation, Evaluation and Technical Outcomes
**Subject:** Security and Privacy for Machine Learning (Semester VII)  
**Student:** Shreya Jadhav | **Roll Number:** 16014223077 | **Batch:** B3 | **Programme:** Honours in Applied Cyber Security  
**Total Marks:** 20 | **Target Dataset:** ULB Credit Card Fraud Detection (284,807 transactions, 492 fraud cases)

---
### IA 2 Marking Rubric Alignment (5 Stages × 4 Marks = 20 Marks)
1. **Problem Refinement & Objectives (4 Marks):** Addresses severe class imbalance (0.1727%), validation-only threshold tuning ($	au^*=0.39$), and multi-seed uncertainty.
2. **Integration of Course Concepts (4 Marks):** Complete coverage across all 5 syllabus modules (MIA, Central DP & DP-SGD, Local DP, PPDM Microaggregation, Adversarial Compressive Privacy).
3. **Dataset Finalisation & Individual Implementation (4 Marks):** Verified real Kaggle dataset (SHA-256 hash, 150MB, zero nulls), leak-free `RobustScaler` on training split.
4. **Evaluation, Comparison & Critical Analysis (4 Marks):** PR-AUC, Confusion Matrix, comparative MIA across private and non-private models, Opacus DP-SGD RDP accounting.
5. **Final Technical Documentation & Research/IP (4 Marks):** Comprehensive 24-section report, reproducibility instructions, resource profiling, and viva preparation handbook.


## 0. Environment Setup & Dependency Verification
Run this cell in **Google Colab** or locally in VS Code. It installs `diffprivlib`, `torch`, `opacus`, and verified dependencies.


In [ ]:
# Environment configuration helper
try:
    import google.colab
    IN_COLAB = True
    !pip install -q diffprivlib opacus torch scikit-learn pandas numpy matplotlib seaborn psutil
except ImportError:
    IN_COLAB = False

import os
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import opacus
import diffprivlib
from configs.config import RANDOM_SEED, MULTI_SEEDS, DATASET_SHA256

print(f"Colab Environment : {IN_COLAB}")
print(f"Project Root      : {PROJECT_ROOT}")
print(f"Torch Version     : {torch.__version__}")
print(f"Opacus Version    : {opacus.__version__}")
print(f"diffprivlib       : {diffprivlib.__version__}")
print(f"Random Seeds      : {MULTI_SEEDS}")


## 1. Dataset Provenance & Diagnostic Inspection
We load the genuine **ULB Credit Card Fraud Detection** dataset (284,807 transactions).


In [ ]:
from src.data_preprocessing import load_dataset, inspect_dataset, split_data, preprocess_features

df, provenance_tag = load_dataset()
inspection = inspect_dataset(df)

print(f"Dataset Provenance      : {provenance_tag}")
print(f"Total Transactions      : {inspection['total_records']:,}")
print(f"Fraudulent Transactions : {inspection['class_distribution']['fraudulent_1']} ({inspection['class_distribution']['fraud_prevalence_percentage']}%)")
print(f"Legitimate Transactions : {inspection['class_distribution']['legitimate_0']:,}")
print(f"Missing Values          : {inspection['total_missing_values']}")
print(f"Duplicate Transactions  : {inspection['duplicate_records']}")

df.head(3)


## 2. Anti-Leakage Partitioning & Scaling
- **Stratified Split:** 70% Train (199,364 rows, 345 frauds), 10% Validation (28,481 rows, 49 frauds), 20% Test (56,962 rows, 98 frauds).
- **Leak-Free Scaling:** `RobustScaler` fitted **strictly on `X_train`**. Validation and test partitions are transformed using training statistics.


In [ ]:
train_df, test_df, val_df, split_meta = split_data(df, method="stratified", random_state=MULTI_SEEDS[0])
X_train, X_test, X_val, y_train, y_test, y_val, feature_names, scaler = preprocess_features(
    train_df, test_df, val_df
)

print(f"Training partition   : {len(X_train):,} rows ({y_train.sum()} fraud cases, {y_train.mean()*100:.4f}%)")
print(f"Validation partition : {len(X_val):,} rows ({y_val.sum()} fraud cases, {y_val.mean()*100:.4f}%)")
print(f"Testing partition    : {len(X_test):,} rows ({y_test.sum()} fraud cases, {y_test.mean()*100:.4f}%)")
print(f"Feature count        : {len(feature_names)}")


## 3. Experiment A: Baseline Random Forest & Validation-Only Threshold Tuning
### Addressing the IA 1 Limitation:
In IA 1, at the default threshold ($	au = 0.50$), minority fraud recall suffered. For IA 2, we implement **validation-only threshold tuning**:
- Search grid: $	au \in [0.01, 0.99]$ evaluated strictly on `X_val` / `y_val`.
- The optimal threshold $	au^* = 0.39$ is then evaluated on the unseen test set `X_test` / `y_test`.


In [ ]:
from src.baseline_model import BaselineFraudDetector
from src.evaluation import find_optimal_threshold, compute_classification_metrics, plot_confusion_matrix, plot_precision_recall_curve_tuned

rf = BaselineFraudDetector()
rf.fit(X_train, y_train, feature_names=feature_names)

# Predict probabilities
p_val = rf.predict_proba(X_val)[:, 1]
p_test = rf.predict_proba(X_test)[:, 1]

# Tune threshold strictly on validation split
best_thresh, val_f1, _ = find_optimal_threshold(y_val, p_val, metric="f1")

# Predictions on unseen test set
y_pred_default = (p_test >= 0.50).astype(int)
y_pred_tuned = (p_test >= best_thresh).astype(int)

m_default = compute_classification_metrics(y_test, y_pred_default, p_test, "Baseline_DefaultThresh", 0.50)
m_tuned = compute_classification_metrics(y_test, y_pred_tuned, p_test, "Baseline_TunedThresh", best_thresh)

print(f"Default Threshold (0.50): Prec={m_default['precision']:.4f} | Rec={m_default['recall']:.4f} | F1={m_default['f1_score']:.4f} | PR-AUC={m_default['pr_auc']:.4f}")
print(f"Tuned Threshold  ({best_thresh:.2f}): Prec={m_tuned['precision']:.4f} | Rec={m_tuned['recall']:.4f} | F1={m_tuned['f1_score']:.4f} | PR-AUC={m_tuned['pr_auc']:.4f}")


In [ ]:
# Visual Diagnostic: PR Curve with Tuned Threshold Point
plot_precision_recall_curve_tuned(
    y_test, p_test,
    tuned_threshold=best_thresh,
    tuned_precision=m_tuned['precision'],
    tuned_recall=m_tuned['recall'],
    title=f"Precision-Recall Curve with Validation-Tuned Operating Point ($\tau^*={best_thresh:.2f}$)",
    save_path=None
)
plt.show()


## 4. Experiment B (Module 1): Comparative Membership Inference Attack
We evaluate shadow-model MIA against **both** non-private and differentially private models ($\epsilon \in \{0.1, 1.0, 10.0\}$).


In [ ]:
from src.membership_inference import run_comparative_mia_experiment

# Subsample 10,000 records for fast shadow training in notebook
idx_mia = np.random.RandomState(42).choice(len(X_train), 10000, replace=False)
mia_out = run_comparative_mia_experiment(X_train[idx_mia], y_train[idx_mia], save_outputs=True)

display(mia_out["comparison_table"][["model_evaluated", "attack_auc", "attack_accuracy", "attacker_advantage"]])


## 5. Experiment C (Module 2): Central Differential Privacy & Opacus DP-SGD
### Theoretical Foundations:
- **IBM `diffprivlib`:** Pure $\epsilon$-DP Logistic Regression via objective perturbation.
- **PyTorch & Opacus DP-SGD:** Per-sample gradient clipping ($C=1.0$), Gaussian noise addition, and Rényi Differential Privacy (RDP) accounting.


In [ ]:
from src.differential_privacy import run_dp_privacy_utility_sweep, train_dp_sgd_opacus

# 1. diffprivlib sweep
dp_sweep_df, dp_summary = run_dp_privacy_utility_sweep(
    X_train, y_train, X_test, y_test, epsilons=[0.1, 1.0, 10.0], data_norm=5.0
)
display(dp_sweep_df[["experiment_name", "epsilon", "precision", "recall", "f1_score", "pr_auc"]])

# 2. PyTorch Opacus DP-SGD
dp_sgd_m, ref_nn_m = train_dp_sgd_opacus(
    X_train, y_train, X_test, y_test,
    target_epsilon=5.0, target_delta=1e-5, epochs=3, batch_size=512
)
print(f"Non-Private NN Ref : Prec={ref_nn_m['precision']:.4f} | Rec={ref_nn_m['recall']:.4f} | F1={ref_nn_m['f1_score']:.4f} | PR-AUC={ref_nn_m['pr_auc']:.4f}")
print(f"Opacus DP-SGD (eps={dp_sgd_m['achieved_epsilon']:.2f}): Prec={dp_sgd_m['precision']:.4f} | Rec={dp_sgd_m['recall']:.4f} | F1={dp_sgd_m['f1_score']:.4f} | PR-AUC={dp_sgd_m['pr_auc']:.4f}")


## 6. Experiment D (Module 3): Local Differential Privacy (Bounded Laplace on Amount)
Demonstrates client-side randomization where each cardholder perturbs their transaction `Amount` before sending it to an untrusted central aggregator.


In [ ]:
from src.local_differential_privacy import run_ldp_demonstration

amt_idx = feature_names.index("Amount") if "Amount" in feature_names else 0
ldp_summary = run_ldp_demonstration(
    X_train, y_train, X_test, y_test,
    feature_index=amt_idx, feature_name="Amount",
    epsilons=[0.5, 1.0, 5.0], save_outputs=True
)

ldp_df = pd.DataFrame(ldp_summary["experiments"])
display(ldp_df[["experiment_name", "epsilon", "wasserstein_distance", "feature_mse", "f1_score", "pr_auc"]])


## 7. Experiment E (Module 4): Privacy-Preserving Data Mining (Continuous Microaggregation)
### The PCA Fallacy:
PCA is an orthogonal linear projection ($Z=XW$) designed to maximize variance. It contains **zero calibrated noise** and provides **no differential privacy guarantee**.  
We demonstrate **Continuous Microaggregation** ($k \in \{5, 10\}$) and measure Information Loss.


In [ ]:
from src.privacy_preserving_mining import run_ppdm_demonstration

n_sub = min(len(X_train), 25000)
idx_sub = np.random.RandomState(42).choice(len(X_train), n_sub, replace=False)

ppdm_summary = run_ppdm_demonstration(
    X_train[idx_sub], y_train[idx_sub], X_test[:5000], y_test[:5000],
    k_values=[5, 10], save_outputs=True
)

ppdm_df = pd.DataFrame(ppdm_summary["experiments"])
display(ppdm_df[["experiment_name", "k_anonymity", "information_loss_pct", "precision", "recall", "f1_score"]])


## 8. Experiment F (Module 5): Adversarial Compressive Privacy Autoencoder
### Information Bottleneck Optimization:
$$\min_{\theta_E, \theta_T} \max_{\theta_A} \left[ \mathcal{L}_{task}(Y, T(Z)) - \lambda \mathcal{L}_{adv}(S, A(Z)) \right]$$
Where $S$ is a justified sensitive proxy attribute (High-Value Transaction Bracket > 85th percentile), $Y$ is fraud detection, and $Z$ is the latent representation.


In [ ]:
from src.compressive_privacy import run_compressive_privacy_demonstration

comp_summary = run_compressive_privacy_demonstration(
    X_train, y_train, X_test, y_test,
    feature_names=feature_names,
    latent_dimensions=[4, 8],
    save_outputs=True
)

comp_df = pd.DataFrame(comp_summary["experiments"])
display(comp_df[["experiment_name", "latent_dimension", "lambda_adversary", "f1_score", "adversary_sensitive_auc", "pr_auc"]])


## 9. Consolidated Evaluation Table & Academic Viva Summary


In [ ]:
from src.evaluation import save_metrics_summary

all_records = (
    [m_default, m_tuned] +
    dp_summary["dp_sweep_results"] +
    [ref_nn_m, dp_sgd_m] +
    ldp_summary["experiments"] +
    ppdm_summary["experiments"] +
    comp_summary["experiments"]
)

summary_table = save_metrics_summary(all_records)
display(summary_table[["Experiment", "Threshold", "Precision", "Recall", "F1-Score", "PR-AUC", "Privacy/Attack Metric"]])
print("Evaluation artifacts persisted to results/experiments_comparison.csv")
